<h1> Feature Extraction using chatGPT and the OpenAI Python API library

In [1]:
import openai
from openai import OpenAI
import pandas as pd

In [2]:
# Prepare the connection through the OpenAI Python API library. Credentials will be read from the environment file by default.
client = OpenAI()

In [3]:
SAMPLE_SIZE = 100
FILE_PATH = f"../data/spotify_reviews_post-2023_{SAMPLE_SIZE}.json"

# Fetch relevant data
df = pd.read_csv('../data/spotify_reviews.csv')

# Filter and format relevant data
df = df.drop(columns=['reviewId', 'userName', 'score', 'thumbsUpCount', 'reviewCreatedVersion'])
df['at'] = pd.to_datetime(df['at'])
df = df[df['at'] > '2023-01-01']
df.sample(SAMPLE_SIZE)
 
# Save data as JSON file
df.to_json(FILE_PATH, orient='records')

In [4]:
system_prompt = """
                You are a helpful assistant analyzing Spotify app reviews. 
                Be precise and focus on specific features which are highlighted in the reviews. 
                Limit your response to the data available. 
                Do not provide general information.
                Provide your response in a suitable markdown format. 
                """

In [5]:
# Create the Assistant
print("Creating assistant...")
assistant = client.beta.assistants.create(
    name="Spotify Review Analyzer",
    instructions=system_prompt,
    model="gpt-3.5-turbo",
    tools=[{"type": "file_search"}],
    temperature=0
)

# Create a vector store to store the data
print("Creating vector store...")
vector_store = client.beta.vector_stores.create(name="Spotify Review Vector Store ({SAMPLE_SIZE})")

# Prepare files for upload to OpenAI
file_paths = [FILE_PATH]
file_streams = [open(path, "rb") for path in file_paths]

# Use SDK helper to upload the files, add them to the vector store and poll the status of the file batch for completion.
print("Uploading files...")
file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)

# You can print the status and the file counts of the batch to see the result of this operation.
print("Finished uploading files.")
print(file_batch.status)
print(file_batch.file_counts)

# Update the assistant with the vector store
print("Updating assistant with vector store...")
assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)

Creating assistant...
Creating vector store...
Uploading files...
Finished uploading files.
completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)
Updating assistant with vector store...


In [12]:
# Create the Thread
thread = openai.beta.threads.create()

# Add individual review to the thread
def add_user_message(message):
    openai.beta.threads.messages.create(
        thread_id=thread.id,
        role="user",
        content=message
    )

# Function to run the assistant on the thread
def run_assistant():
    run = openai.beta.threads.runs.create(
        thread_id=thread.id,
        assistant_id=assistant.id
    )
    return run

# Function to retrieve and process assistant's response
def get_assistant_response(run_id):
    messages = openai.beta.threads.messages.list(
        thread_id=thread.id
    )
    response_message = messages.data[0]
    return response_message.content[0].text.value

# Function to add user message and run the assistant
def add_user_message_and_run(user_prompt):
    add_user_message(user_prompt)
    run = run_assistant()
    while run.status != "completed":
        run = openai.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    assistant_response = get_assistant_response(run.id)
    return assistant_response

In [13]:
prompt_likes = """
                Please extract the top 10 specific features that users like the most about the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_likes))

Based on the reviews data, the top 10 specific features that users like the most about the Spotify app are:

1. **Good choice of genres for all ages and tastes**
2. **Excellent sound quality**
3. **Intuitive playlists that anticipate user preferences**
4. **Reasonable pricing**
5. **Ease of navigation**
6. **Family sharing experience**
7. **Huge selection of music and podcasts**
8. **Personalized music listening experience**
9. **AI DJ feature**
10. **Ability to create and manage playlists efficiently**

These features were highlighted positively by users in the reviews of the Spotify app【4:0†source】【4:1†source】【4:2†source】【4:3†source】.


In [14]:
prompt_dislikes = """
                    Please extract the top 10 specific features that users dislike the most about the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_dislikes))

Based on the reviews data, the top 10 specific features that users dislike the most about the Spotify app are:

1. **Slow, glitchy, and crashing app performance**
2. **Issues with music playback, such as lagging, difficulty skipping songs, and incorrect song selections**
3. **Intrusive ads and unwanted content like audiobooks and podcasts**
4. **Problems with offline music playback and downloaded music not showing**
5. **Difficulties with the shuffle feature and multiple device playback**
6. **Removal of the heart like button and changes in the user interface**
7. **Limited features for free users, such as song skips and specific song selection**
8. **Technical issues with storage location settings and frequent app updates affecting usability**
9. **Smart shuffle suggestions and inability to turn off certain features**
10. **Changes in app layout, lyrics availability, and overall user experience**

These features were highlighted negatively by users in the reviews of the Spotify app【8:

In [15]:
prompt_wants = """
                Please extract the top 10 specific features that users want to see in the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_wants))

Based on the reviews data, the top 10 specific features that users want to see in the Spotify app are:

1. **Speed changer feature**
2. **Improved lyrics loading**
3. **Removal of the DJ feature for listening to songs**
4. **Enhanced options for going back in playing music**
5. **Ability to turn off shuffle without premium**
6. **Desire for a "Shazam" type option to identify songs**
7. **Reintroduction of the heart like button in playlists**
8. **Parallel apps for multiple Spotify accounts**
9. **Playlist folders creation on the Android app**
10. **Lock screen widget functionality**

These features were requested by users in the reviews of the Spotify app【12:0†source】【12:1†source】【12:2†source】【12:3†source】【12:4†source】.


In [16]:
prompt_bugs = """
                Please extract the top 10 specific bugs that users have reported in the Spotify app based on the reviews data.
                """

print(add_user_message_and_run(prompt_bugs))

Based on the reviews data, the top 10 specific bugs that users have reported in the Spotify app are:

1. **Songs randomly stopping during playback**
2. **Issues with playing music in the background**
3. **Smart shuffle feature turning on automatically**
4. **App crashing on startup**
5. **Podcast page loading slowly**
6. **Problems with offline mode and internet connection detection**
7. **Notification widget displaying incorrect song information**
8. **Bluetooth-related stuttering and skipping during playback**
9. **Repeated ads and limitations on song selection without premium**
10. **Playlist function behaving unexpectedly, songs not progressing to the next one**

These bugs were reported by users in the reviews of the Spotify app【16:0†source】【16:1†source】【16:2†source】【16:3†source】【16:4†source】.


In [17]:
prompt_other = """
                Please extract other specific insights from the Spotify app reviews data that are important and which have not been covered by the previous prompts.
                """

print(add_user_message_and_run(prompt_other))

Some specific insights from the Spotify app reviews data that have not been covered in previous prompts include:

1. Users have reported issues with the stability of the full app on Android, mentioning that it closes itself for no reason and is memory-intensive, leading them to use the lite version instead【20:1†source】.
2. There are complaints about the app being limiting on mobile without premium, especially compared to the PC and console versions. Users find the ads annoying and express frustration over limited skips and song selection without premium【20:1†source】.
3. Users appreciate the ad-free version of Spotify, mentioning that it helps them feel good during their workday【20:2†source】.
4. Some users express disappointment with the app's UI, mentioning issues like the inability to play specific songs easily and changes that make the app less intuitive and more frustrating to use【20:3†source】.
5. Users have experienced random app closures on their phones, leading to frustration and